# OOP and Pythonic Idioms

Python has a distinctive style—'Pythonic' code is clean, readable, and leverages the language's unique features. This notebook covers classes, dataclasses, context managers, and the idioms that separate Python beginners from experienced practitioners.

## Learning Objectives

- Create classes with proper `__init__`, `__repr__`, and `__str__` methods
- Use dataclasses for cleaner data-holding classes
- Implement context managers with the `with` statement
- Use `pathlib.Path` for all file operations
- Apply Pythonic idioms: enumerate, zip, f-strings, and more

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from itertools import zip_longest
from contextlib import contextmanager
import tempfile
import time

## Object-Oriented Programming Fundamentals

Before diving into advanced OOP features and Pythonic idioms, let's review the core concepts of object-oriented programming in Python.

### What is Object-Oriented Programming?

OOP is a programming paradigm that organizes code around "objects" - instances of classes that combine data (attributes) and behavior (methods). The key benefits are:

- **Encapsulation**: Bundle data and methods that work on that data
- **Reusability**: Create classes once, instantiate many times
- **Modularity**: Organize code into logical, self-contained units
- **Abstraction**: Hide complex implementation details

### Creating Your First Class

In [ ]:
# Simple class definition
class Dog:
    """A simple Dog class."""
    pass

# Create an instance (object) of the class
my_dog = Dog()
print(f"Created a dog: {my_dog}")
print(f"Type: {type(my_dog)}")

### Attributes (Instance Variables)

Attributes store data specific to each instance of a class.

In [ ]:
# Class with attributes
class Dog:
    """A dog with a name and age."""
    
    def __init__(self, name, age):
        """Initialize the dog with name and age."""
        self.name = name  # Instance attribute
        self.age = age    # Instance attribute

# Create instances
dog1 = Dog("Buddy", 3)
dog2 = Dog("Max", 5)

print(f"Dog 1: {dog1.name} is {dog1.age} years old")
print(f"Dog 2: {dog2.name} is {dog2.age} years old")

> **Note: The `self` parameter**
> 
> `self` represents the instance of the class. It must be the first parameter of instance methods, but you don't pass it explicitly when calling the method.

### Methods (Instance Functions)

Methods are functions that belong to a class and can access/modify instance attributes.

In [ ]:
class Dog:
    """A dog that can perform actions."""
    
    def __init__(self, name, age):
        self.name = name
        self.age = age
    
    def bark(self):
        """Make the dog bark."""
        print(f"{self.name} says Woof!")
    
    def get_info(self):
        """Return information about the dog."""
        return f"{self.name} is {self.age} years old"
    
    def have_birthday(self):
        """Increment the dog's age."""
        self.age += 1
        print(f"Happy birthday {self.name}! Now {self.age} years old.")

# Create and use a dog
my_dog = Dog("Buddy", 3)
my_dog.bark()
print(my_dog.get_info())
my_dog.have_birthday()

### Class Attributes vs Instance Attributes

Class attributes are shared by all instances, while instance attributes are unique to each instance.

In [ ]:
class Dog:
    """Dog class with both class and instance attributes."""
    
    # Class attribute (shared by all instances)
    species = "Canis familiaris"
    
    def __init__(self, name, age):
        # Instance attributes (unique to each instance)
        self.name = name
        self.age = age

dog1 = Dog("Buddy", 3)
dog2 = Dog("Max", 5)

print(f"Dog 1 species: {dog1.species}")
print(f"Dog 2 species: {dog2.species}")
print(f"All dogs are {Dog.species}")

# Instance attributes are different
print(f"\nDog 1 name: {dog1.name}")
print(f"Dog 2 name: {dog2.name}")

### Inheritance

Inheritance allows you to create a new class based on an existing class.

In [ ]:
# Parent class
class Animal:
    """A general animal class."""
    
    def __init__(self, name):
        self.name = name
    
    def speak(self):
        """Generic speak method."""
        print(f"{self.name} makes a sound")

# Child class inheriting from Animal
class Dog(Animal):
    """A dog is a type of animal."""
    
    def speak(self):  # Override parent method
        print(f"{self.name} says Woof!")

class Cat(Animal):
    """A cat is a type of animal."""
    
    def speak(self):  # Override parent method
        print(f"{self.name} says Meow!")

# Create instances
dog = Dog("Buddy")
cat = Cat("Whiskers")

dog.speak()
cat.speak()

In [ ]:
# Using super() to call parent class methods
class Animal:
    def __init__(self, name, age):
        self.name = name
        self.age = age
    
    def get_info(self):
        return f"{self.name} is {self.age} years old"

class Dog(Animal):
    def __init__(self, name, age, breed):
        # Call parent class __init__
        super().__init__(name, age)
        self.breed = breed
    
    def get_info(self):
        # Extend parent method
        base_info = super().get_info()
        return f"{base_info} and is a {self.breed}"

dog = Dog("Buddy", 3, "Golden Retriever")
print(dog.get_info())

### Encapsulation and "Private" Attributes

Python uses naming conventions to indicate private attributes (prefix with underscore).

In [ ]:
class BankAccount:
    """A bank account with private balance."""
    
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance  # "Private" attribute (by convention)
    
    def deposit(self, amount):
        """Deposit money."""
        if amount > 0:
            self._balance += amount
            print(f"Deposited ${amount}. New balance: ${self._balance}")
        else:
            print("Deposit amount must be positive")
    
    def withdraw(self, amount):
        """Withdraw money."""
        if 0 < amount <= self._balance:
            self._balance -= amount
            print(f"Withdrew ${amount}. New balance: ${self._balance}")
        else:
            print("Invalid withdrawal amount")
    
    def get_balance(self):
        """Get current balance."""
        return self._balance

# Use the account
account = BankAccount("Alice", 100)
account.deposit(50)
account.withdraw(30)
print(f"Current balance: ${account.get_balance()}")

# You can still access _balance, but the underscore signals "don't touch"
print(f"Direct access (not recommended): ${account._balance}")

### Properties (Getters and Setters)

Properties provide controlled access to attributes.

In [ ]:
class Circle:
    """A circle with radius property."""
    
    def __init__(self, radius):
        self._radius = radius
    
    @property
    def radius(self):
        """Get the radius."""
        return self._radius
    
    @radius.setter
    def radius(self, value):
        """Set the radius with validation."""
        if value > 0:
            self._radius = value
        else:
            raise ValueError("Radius must be positive")
    
    @property
    def area(self):
        """Calculate area (read-only property)."""
        import math
        return math.pi * self._radius ** 2
    
    @property
    def circumference(self):
        """Calculate circumference (read-only property)."""
        import math
        return 2 * math.pi * self._radius

# Use the circle
circle = Circle(5)
print(f"Radius: {circle.radius}")
print(f"Area: {circle.area:.2f}")
print(f"Circumference: {circle.circumference:.2f}")

# Change radius
circle.radius = 10
print(f"\nNew radius: {circle.radius}")
print(f"New area: {circle.area:.2f}")

### Class Methods and Static Methods

In [ ]:
class Date:
    """A simple date class."""
    
    def __init__(self, year, month, day):
        self.year = year
        self.month = month
        self.day = day
    
    @classmethod
    def from_string(cls, date_string):
        """Create a Date from a string (alternative constructor)."""
        year, month, day = map(int, date_string.split('-'))
        return cls(year, month, day)  # cls refers to the class
    
    @staticmethod
    def is_valid_date(year, month, day):
        """Check if a date is valid (doesn't need instance or class)."""
        return 1 <= month <= 12 and 1 <= day <= 31
    
    def __str__(self):
        return f"{self.year}-{self.month:02d}-{self.day:02d}"

# Regular constructor
date1 = Date(2024, 4, 7)
print(f"Date 1: {date1}")

# Class method (alternative constructor)
date2 = Date.from_string("2024-12-25")
print(f"Date 2: {date2}")

# Static method (utility function)
print(f"Is 2024-2-30 valid? {Date.is_valid_date(2024, 2, 30)}")
print(f"Is 2024-2-15 valid? {Date.is_valid_date(2024, 2, 15)}")

### Practice Exercises: OOP Fundamentals

Try these exercises to practice object-oriented programming.

#### Exercise A: Rectangle Class

Create a `Rectangle` class with:
- `__init__(width, height)` to initialize dimensions
- A method `area()` that returns the area
- A method `perimeter()` that returns the perimeter
- A method `is_square()` that returns True if width equals height

In [ ]:
# Write your Rectangle class here
# YOUR CODE HERE

In [ ]:
# Solution to Exercise A
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height
    
    def area(self):
        return self.width * self.height
    
    def perimeter(self):
        return 2 * (self.width + self.height)
    
    def is_square(self):
        return self.width == self.height

rect = Rectangle(5, 10)
print(f"Area: {rect.area()}, Perimeter: {rect.perimeter()}, Square: {rect.is_square()}")

#### Exercise B: Student Class with Grades

Create a `Student` class with:
- `__init__(name)` to initialize with a name
- An empty list of grades
- A method `add_grade(grade)` to add a grade (0-100)
- A method `get_average()` that returns the average grade
- A method `get_letter_grade()` that returns 'A' (90+), 'B' (80-89), 'C' (70-79), 'D' (60-69), or 'F' (<60)

In [ ]:
# Write your Student class here
# YOUR CODE HERE

In [ ]:
# Solution to Exercise B
class Student:
    def __init__(self, name):
        self.name = name
        self.grades = []
    
    def add_grade(self, grade):
        if 0 <= grade <= 100:
            self.grades.append(grade)
    
    def get_average(self):
        return sum(self.grades) / len(self.grades) if self.grades else 0
    
    def get_letter_grade(self):
        avg = self.get_average()
        if avg >= 90: return 'A'
        elif avg >= 80: return 'B'
        elif avg >= 70: return 'C'
        elif avg >= 60: return 'D'
        else: return 'F'

student = Student("Alice")
student.add_grade(85)
student.add_grade(92)
print(f"Average: {student.get_average():.1f}, Grade: {student.get_letter_grade()}")

#### Exercise C: Book and Library Classes

Create two classes:

1. `Book` class with:
   - `__init__(title, author, isbn)` to initialize
   - `__str__()` to return a string like "Title by Author (ISBN: ...)"

2. `Library` class with:
   - `__init__()` to initialize an empty list of books
   - `add_book(book)` to add a Book to the library
   - `find_books_by_author(author)` to return a list of books by that author
   - `total_books` property that returns the number of books

In [ ]:
# Write your Book and Library classes here
# YOUR CODE HERE

In [ ]:
# Solution to Exercise C
class Book:
    def __init__(self, title, author, isbn):
        self.title = title
        self.author = author
        self.isbn = isbn
    
    def __str__(self):
        return f"{self.title} by {self.author} (ISBN: {self.isbn})"

class Library:
    def __init__(self):
        self._books = []
    
    def add_book(self, book):
        self._books.append(book)
    
    def find_books_by_author(self, author):
        return [book for book in self._books if book.author == author]
    
    @property
    def total_books(self):
        return len(self._books)

library = Library()
library.add_book(Book("1984", "George Orwell", "123"))
print(f"Total books: {library.total_books}")

#### Exercise D: Vehicle Inheritance

Create a class hierarchy:

1. `Vehicle` (parent class) with:
   - `__init__(make, model, year)`
   - `get_info()` method that returns vehicle information

2. `Car` (child of Vehicle) with:
   - Additional `num_doors` attribute
   - Override `get_info()` to include number of doors

3. `Motorcycle` (child of Vehicle) with:
   - Additional `has_sidecar` attribute (boolean)
   - Override `get_info()` to include sidecar information

In [ ]:
# Write your Vehicle, Car, and Motorcycle classes here
# YOUR CODE HERE

In [ ]:
# Solution to Exercise D
class Vehicle:
    def __init__(self, make, model, year):
        self.make = make
        self.model = model
        self.year = year
    
    def get_info(self):
        return f"{self.year} {self.make} {self.model}"

class Car(Vehicle):
    def __init__(self, make, model, year, num_doors):
        super().__init__(make, model, year)
        self.num_doors = num_doors
    
    def get_info(self):
        return f"{super().get_info()} ({self.num_doors}-door)"

car = Car("Toyota", "Camry", 2024, 4)
print(car.get_info())

---

## Advanced OOP Features and Pythonic Idioms

Now that you're comfortable with OOP basics, let's explore advanced Python-specific features.

## Classes: `__init__`, `__repr__`, `__str__`

Every class should implement at least `__init__` and `__repr__`. `__str__` is optional but useful for user-facing output.

In [ ]:
class Point:
    """A 2D point with x and y coordinates."""
    
    def __init__(self, x: float, y: float):
        """Initialize point with x and y coordinates."""
        self.x = x
        self.y = y
    
    def __repr__(self) -> str:
        """Developer-friendly representation (unambiguous)."""
        return f"Point(x={self.x}, y={self.y})"
    
    def __str__(self) -> str:
        """User-friendly representation."""
        return f"({self.x}, {self.y})"
    
    def distance_from_origin(self) -> float:
        """Calculate distance from origin (0, 0)."""
        return (self.x ** 2 + self.y ** 2) ** 0.5

p = Point(3, 4)
print(f"repr: {repr(p)}")  # For developers/debugging
print(f"str: {str(p)}")    # For users
print(f"Distance: {p.distance_from_origin()}")

In [ ]:
# A more complete class example
class DataRecord:
    """A record with ID, name, and optional metadata."""
    
    def __init__(self, id: int, name: str, **metadata):
        self.id = id
        self.name = name
        self.metadata = metadata
        self._created_at = time.time()
    
    def __repr__(self):
        return f"DataRecord(id={self.id}, name={self.name!r}, **{self.metadata})"
    
    def __str__(self):
        return f"Record #{self.id}: {self.name}"
    
    def __eq__(self, other):
        """Two records are equal if they have the same ID."""
        if not isinstance(other, DataRecord):
            return False
        return self.id == other.id
    
    def __hash__(self):
        """Enable use in sets and as dict keys."""
        return hash(self.id)

r1 = DataRecord(1, "Alice", department="Engineering")
r2 = DataRecord(2, "Bob", department="Marketing", level="Senior")
print(r1)
print(repr(r2))

## Dataclasses (Python 3.7+)

Dataclasses automatically generate `__init__`, `__repr__`, `__eq__`, and more. They're perfect for data-holding classes that are common in data science.

In [ ]:
@dataclass
class PointDC:
    """A 2D point using dataclass."""
    x: float
    y: float
    
    def distance_from_origin(self) -> float:
        return (self.x ** 2 + self.y ** 2) ** 0.5

p = PointDC(3, 4)
print(p)  # Auto-generated __repr__
print(f"Distance: {p.distance_from_origin()}")

# Auto-generated __eq__
print(f"PointDC(3, 4) == PointDC(3, 4): {PointDC(3, 4) == PointDC(3, 4)}")

In [ ]:
@dataclass
class Employee:
    """Employee record with defaults and computed fields."""
    id: int
    name: str
    department: str = "General"
    salary: float = 50000.0
    skills: list = field(default_factory=list)  # For mutable defaults!
    
    @property
    def monthly_salary(self) -> float:
        """Computed property."""
        return self.salary / 12

e1 = Employee(1, "Alice", "Engineering", 100000)
e2 = Employee(2, "Bob")  # Uses defaults

print(e1)
print(f"Alice's monthly salary: ${e1.monthly_salary:,.2f}")
print(e2)

# Skills list is separate for each instance (thanks to field(default_factory=list))
e1.skills.append("Python")
print(f"Alice's skills: {e1.skills}")
print(f"Bob's skills: {e2.skills}")  # Not affected!

> **Gotcha: Mutable defaults in dataclasses**  
> Use `field(default_factory=list)` instead of `field(default=[])` for mutable defaults—same reason as regular function defaults.

In [ ]:
# Frozen dataclass (immutable)
@dataclass(frozen=True)
class Config:
    """Immutable configuration."""
    host: str
    port: int
    debug: bool = False

config = Config("localhost", 8080)
print(config)

# Can be used as dict key (it's hashable)
settings = {config: "default settings"}
print(f"Lookup: {settings[config]}")

# Trying to modify raises an error
try:
    config.port = 3000
except Exception as e:
    print(f"Error: {e}")

## Context Managers with `with`

Context managers ensure cleanup happens even if an exception occurs. They're essential for file handling, database connections, and resource management.

In [ ]:
# The classic example: file handling
# BAD: May leave file open if exception occurs
# f = open('file.txt')
# data = f.read()
# f.close()  # Might not run if exception above!

# GOOD: with statement guarantees file is closed
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write("Hello, context managers!")
    temp_path = f.name

print(f"File written to: {temp_path}")
# File is automatically closed here

In [ ]:
# Creating a context manager with a class
class Timer:
    """Context manager that times a block of code."""
    
    def __enter__(self):
        self.start = time.perf_counter()
        return self  # The value bound to 'as' variable
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time.perf_counter()
        self.elapsed = self.end - self.start
        print(f"Elapsed time: {self.elapsed:.4f} seconds")
        return False  # Don't suppress exceptions

with Timer():
    # Some computation
    total = sum(x**2 for x in range(1_000_000))

In [ ]:
# Creating a context manager with @contextmanager decorator
@contextmanager
def timer_cm():
    """Context manager that times code execution."""
    start = time.perf_counter()
    yield  # Control returns to the 'with' block here
    end = time.perf_counter()
    print(f"Elapsed: {end - start:.4f}s")

with timer_cm():
    total = sum(x**2 for x in range(1_000_000))

In [ ]:
# Practical example: temporary directory
@contextmanager
def working_directory(path):
    """Temporarily change to a directory, then change back."""
    import os
    old_dir = os.getcwd()
    try:
        os.chdir(path)
        yield
    finally:
        os.chdir(old_dir)

# Usage:
# with working_directory('/tmp'):
#     # Do work in /tmp
#     pass
# # Automatically back to original directory

## pathlib.Path — Modern File Operations

Forget `os.path`! `pathlib` provides an object-oriented interface for file system paths. It's more readable and cross-platform.

In [ ]:
# Creating paths
p = Path('/Users/alice/documents')
print(f"Path: {p}")

# Joining paths with / operator
data_path = p / 'data' / 'file.csv'
print(f"Joined: {data_path}")

# Current directory
cwd = Path.cwd()
print(f"Current directory: {cwd}")

# Home directory
home = Path.home()
print(f"Home directory: {home}")

In [ ]:
# Path components
p = Path('/home/user/project/data/train.csv')

print(f"Name: {p.name}")         # train.csv
print(f"Stem: {p.stem}")         # train (name without extension)
print(f"Suffix: {p.suffix}")     # .csv
print(f"Parent: {p.parent}")     # /home/user/project/data
print(f"Parts: {p.parts}")       # ('/', 'home', 'user', ...)

In [ ]:
# Common operations
p = Path.cwd()

# Check existence
print(f"Exists: {p.exists()}")
print(f"Is directory: {p.is_dir()}")
print(f"Is file: {p.is_file()}")

# List directory contents
print("\nFiles in current directory:")
for item in p.iterdir():
    prefix = "[DIR]" if item.is_dir() else "[FILE]"
    print(f"  {prefix} {item.name}")

In [ ]:
# Glob patterns for finding files
# Find all Python files
# py_files = list(Path('.').glob('**/*.py'))

# Reading and writing with Path
test_file = Path(tempfile.gettempdir()) / 'test_pathlib.txt'

# Write text
test_file.write_text("Hello from pathlib!\nLine 2")

# Read text
content = test_file.read_text()
print(f"Content:\n{content}")

# Delete
test_file.unlink()
print(f"File exists after unlink: {test_file.exists()}")

## f-strings and String Formatting

f-strings (formatted string literals) are the modern way to format strings in Python. They're faster and more readable than `.format()` or `%` formatting.

In [ ]:
name = "Alice"
age = 30
salary = 95000.5

# Basic f-string
print(f"Name: {name}, Age: {age}")

# Expressions inside f-strings
print(f"Next year, {name} will be {age + 1}")

# Formatting numbers
print(f"Salary: ${salary:,.2f}")  # Comma separator, 2 decimal places
print(f"Percentage: {0.856:.1%}")  # As percentage

# Padding and alignment
for item in ['apple', 'banana', 'cherry']:
    print(f"{item:>10}")  # Right-align, width 10

In [ ]:
# Debugging with f-strings (Python 3.8+)
x = 10
y = 20
print(f"{x=}, {y=}, {x + y=}")  # Prints: x=10, y=20, x + y=30

# With format specs
pi = 3.14159265359
print(f"{pi=:.2f}")  # pi=3.14

## Enumerate, Zip, and Other Pythonic Patterns

In [ ]:
# enumerate: get index and value together
fruits = ['apple', 'banana', 'cherry']

# BAD
# for i in range(len(fruits)):
#     print(i, fruits[i])

# GOOD
for i, fruit in enumerate(fruits):
    print(f"{i}: {fruit}")

# Start at different index
print("\nStarting at 1:")
for i, fruit in enumerate(fruits, start=1):
    print(f"{i}. {fruit}")

In [ ]:
# zip: iterate over multiple sequences together
names = ['Alice', 'Bob', 'Charlie']
scores = [85, 92, 78]

# BAD
# for i in range(len(names)):
#     print(names[i], scores[i])

# GOOD
for name, score in zip(names, scores):
    print(f"{name}: {score}")

# Create dict from two lists
score_dict = dict(zip(names, scores))
print(f"\nAs dict: {score_dict}")

In [ ]:
# zip_longest: when sequences have different lengths
names = ['Alice', 'Bob', 'Charlie', 'Diana']
scores = [85, 92]

# Regular zip stops at shortest
print("zip (stops at shortest):")
for name, score in zip(names, scores):
    print(f"  {name}: {score}")

# zip_longest fills missing values
print("\nzip_longest (fills missing):")
for name, score in zip_longest(names, scores, fillvalue='N/A'):
    print(f"  {name}: {score}")

## Bad Code → Idiomatic Code Refactoring

Let's transform non-Pythonic code step by step.

In [ ]:
# ORIGINAL: Non-Pythonic code
data = [{'name': 'Alice', 'score': 85}, 
        {'name': 'Bob', 'score': 92},
        {'name': 'Charlie', 'score': 78}]

# Find average score for people with score > 80
sum_score = 0
count = 0
i = 0
while i < len(data):
    if data[i]['score'] > 80:
        sum_score = sum_score + data[i]['score']
        count = count + 1
    i = i + 1

if count > 0:
    avg = sum_score / count
else:
    avg = 0

print("Non-Pythonic result:", avg)

In [ ]:
# STEP 1: Use for loop instead of while with index
sum_score = 0
count = 0
for person in data:
    if person['score'] > 80:
        sum_score += person['score']
        count += 1

avg = sum_score / count if count > 0 else 0
print("Step 1 result:", avg)

In [ ]:
# STEP 2: Use list comprehension to filter
high_scores = [p['score'] for p in data if p['score'] > 80]
avg = sum(high_scores) / len(high_scores) if high_scores else 0
print("Step 2 result:", avg)

In [ ]:
# FINAL: One-liner with generator expression (if you must)
# But the Step 2 version is more readable!
scores = [p['score'] for p in data if p['score'] > 80]
avg = sum(scores) / len(scores) if scores else 0
print(f"Pythonic result: {avg}")

## Exercises

### Exercise 1: Dataclass for Data Science

Create a `Dataset` dataclass that holds a name, a list of feature names, a row count, and optional metadata. Include a method that returns a summary string.

In [ ]:
# Create a Dataset dataclass with:
# - name: str
# - features: List[str]
# - n_rows: int
# - metadata: Dict (default to empty dict)
# - A summary() method that returns a formatted string

# YOUR CODE HERE

In [ ]:
# Solution to Exercise 1
from dataclasses import dataclass, field
from typing import List

@dataclass
class Product:
    name: str
    price: float
    tags: List[str] = field(default_factory=list)
    
    def add_tag(self, tag: str):
        """Add a tag to the product."""
        if tag not in self.tags:
            self.tags.append(tag)
    
    def has_tag(self, tag: str) -> bool:
        """Check if product has a specific tag."""
        return tag in self.tags
    
    def __repr__(self):
        tags_str = ', '.join(self.tags) if self.tags else 'no tags'
        return f"Product('{self.name}', ${self.price:.2f}, tags: {tags_str})"

# Test the class
laptop = Product("Dell XPS", 1299.99)
laptop.add_tag("electronics")
laptop.add_tag("computer")
print(laptop)
print(f"Has 'electronics' tag: {laptop.has_tag('electronics')}")

phone = Product("iPhone 15", 999.99, ["electronics", "mobile"])
print(phone)

### Exercise 2: Context Manager for Database Connection

Create a context manager (using the `@contextmanager` decorator) that simulates a database connection: prints "Connecting...", yields a fake connection object, then prints "Disconnecting...".

In [ ]:
# Create a database_connection context manager
# Usage:
# with database_connection('mydb') as conn:
#     print(f"Using connection: {conn}")

# YOUR CODE HERE

In [ ]:
# Solution to Exercise 2
class DatabaseConnection:
    """Context manager for database connections."""
    
    def __init__(self, db_name):
        self.db_name = db_name
        self.connection = None
    
    def __enter__(self):
        """Establish connection when entering context."""
        print(f"Connecting to database '{self.db_name}'...")
        self.connection = f"Connection to {self.db_name}"
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Close connection when exiting context."""
        print(f"Closing connection to '{self.db_name}'...")
        self.connection = None
        return False  # Don't suppress exceptions
    
    def execute(self, query):
        """Execute a query."""
        if self.connection:
            print(f"Executing: {query}")
            return "Result"
        else:
            raise RuntimeError("Not connected to database")

# Test the context manager
with DatabaseConnection("my_database") as db:
    db.execute("SELECT * FROM users")
    db.execute("INSERT INTO logs VALUES (1, 'test')")
print("Connection automatically closed!")

### Exercise 3: File Processing with pathlib

Write a function that takes a directory path and returns a dictionary mapping file extensions to lists of file names.

In [ ]:
def group_files_by_extension(directory: str) -> dict:
    """
    Group files in directory by their extension.
    
    Returns: {'py': ['main.py', 'utils.py'], 'txt': ['readme.txt'], ...}
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Solution to Exercise 3
from pathlib import Path

def process_log_files(directory_path: str) -> dict:
    """Process all .log files in a directory."""
    log_dir = Path(directory_path)
    
    results = {
        'total_files': 0,
        'total_lines': 0,
        'error_count': 0,
        'warning_count': 0,
        'files': []
    }
    
    # Find all .log files
    for log_file in log_dir.glob('*.log'):
        results['total_files'] += 1
        file_info = {
            'name': log_file.name,
            'size': log_file.stat().st_size,
            'lines': 0,
            'errors': 0,
            'warnings': 0
        }
        
        # Read and process each file
        with log_file.open('r') as f:
            for line in f:
                file_info['lines'] += 1
                results['total_lines'] += 1
                
                line_lower = line.lower()
                if 'error' in line_lower:
                    file_info['errors'] += 1
                    results['error_count'] += 1
                if 'warning' in line_lower:
                    file_info['warnings'] += 1
                    results['warning_count'] += 1
        
        results['files'].append(file_info)
    
    return results

# Example usage (would work if log files existed):
# results = process_log_files('/path/to/logs')
# print(f"Processed {results['total_files']} files")
# print(f"Total errors: {results['error_count']}")
# print(f"Total warnings: {results['warning_count']}")

print("Log file processor function defined successfully!")

### Exercise 4: Refactoring Challenge

Refactor this non-Pythonic code to be clean and idiomatic:

In [ ]:
# ORIGINAL: Non-Pythonic code to refactor
# This code pairs students with their scores and formats them

students = ['Alice', 'Bob', 'Charlie']
scores = [85, 92, 78]

results = []
i = 0
while i < len(students):
    result = students[i] + ": " + str(scores[i])
    results.append(result)
    i = i + 1

for j in range(len(results)):
    print(str(j + 1) + ". " + results[j])

# REFACTORED: Your Pythonic version below
# YOUR CODE HERE

In [ ]:
# Solution to Exercise 4
# Original non-idiomatic code (from the notebook):
# data = [{'name': 'Alice', 'age': 30}, {'name': 'Bob', 'age': 25}, {'name': 'Charlie', 'age': 35}]
# names = []
# i = 0
# while i < len(data):
#     names.append(data[i]['name'])
#     i = i + 1

# Refactored idiomatic code:
data = [
    {'name': 'Alice', 'age': 30},
    {'name': 'Bob', 'age': 25},
    {'name': 'Charlie', 'age': 35}
]

# Idiomatic version 1: List comprehension
names = [person['name'] for person in data]
print("Names:", names)

# Even more idiomatic: using operator.itemgetter
from operator import itemgetter
names_v2 = list(map(itemgetter('name'), data))
print("Names (v2):", names_v2)

# If we want both names and ages:
info = [(person['name'], person['age']) for person in data]
print("Info:", info)

# Or using f-strings for formatted output:
formatted = [f"{p['name']} ({p['age']} years old)" for p in data]
for person_info in formatted:
    print(f"  - {person_info}")